In [1]:
import sys
print(sys.executable)

c:\Users\Araceli\Desktop\IFST 18\Seminario de actualizacion\.venv\Scripts\python.exe


In [2]:
import sys, sklearn, pandas, joblib
print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)

Python: 3.13.7
scikit-learn: 1.9.1


In [3]:
import os
import pandas as pd

URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
os.makedirs("datos", exist_ok=True)

df = pd.read_csv(URL)
df.to_csv("datos/insurance.csv", index=False)
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


El origen del dataset es del libro Machine Learning with R, realizado con datos simulados.

In [4]:
df = df.rename(columns={
    "age": "edad", "sex": "sexo", "bmi": "imc", "children": "hijos",
    "smoker": "fumador", "region": "region", "charges": "costo",
})
df.head()

,edad,sexo,imc,hijos,fumador,region,costo
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Tiene 6 Features, edad, imc, hijos (numericas) y sexo, fumador, region (categoricas).
El target es el costo.

In [5]:
df.shape

(1338, 7)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   edad     1338 non-null   int64  
 1   sexo     1338 non-null   str    
 2   imc      1338 non-null   float64
 3   hijos    1338 non-null   int64  
 4   fumador  1338 non-null   str    
 5   region   1338 non-null   str    
 6   costo    1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [7]:
df.describe().round(1)

,edad,imc,hijos,costo
count,1338.0,1338.0,1338.0,1338.0
mean,39.2,30.7,1.1,13270.4
std,14.0,6.1,1.2,12110.0
min,18.0,16.0,0.0,1121.9
25%,27.0,26.3,0.0,4740.3
50%,39.0,30.4,1.0,9382.0
75%,51.0,34.7,2.0,16639.9
max,64.0,53.1,5.0,63770.4


La mediana del costo es de 9382, mientras que la media es de 13270, la variable mas importante que puede afectar retundamente a esta diferencia es que hay muchos mas no fumadores que fumadores

In [8]:
df.duplicated().sum()

np.int64(1)

In [10]:
df = df.drop_duplicates().reset_index(drop=True)
df.shape

(1337, 7)

In [11]:
df.groupby("fumador")["costo"].mean().round(0)

fumador
no      8441.0
yes    32050.0
Name: costo, dtype: float64

La diferencia de costo entre fumadores y no fumadores es de aproximadamente 24000, es aproximadamente un 3,8 mas caro para una persona fumadora.

In [12]:
from sklearn.model_selection import train_test_split

NUMERICAS = ["edad", "imc", "hijos"]
CATEGORICAS = ["sexo", "fumador", "region"]

X = df[NUMERICAS + CATEGORICAS]   # features
y = df["costo"]                   # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(len(X_train), "filas para entrenar")
print(len(X_test), "filas para probar")

1069 filas para entrenar
268 filas para probar


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preparacion = ColumnTransformer([
    ("num", StandardScaler(), NUMERICAS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS),
])

# Solo para ver qué hace:
preparacion.fit(X_train)
print(preparacion.get_feature_names_out())

['num__edad' 'num__imc' 'num__hijos' 'cat__sexo_female' 'cat__sexo_male'
 'cat__fumador_no' 'cat__fumador_yes' 'cat__region_northeast'
 'cat__region_northwest' 'cat__region_southeast' 'cat__region_southwest']


In [14]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def evaluar(nombre, modelo):
    pipe = Pipeline([("prep", preparacion), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f"{nombre:15} MAE: {mean_absolute_error(y_test, pred):8.0f}   "
          f"RMSE: {root_mean_squared_error(y_test, pred):8.0f}   "
          f"R²: {r2_score(y_test, pred):.3f}")
    return pipe

base   = evaluar("Línea base", DummyRegressor())
lineal = evaluar("Reg. lineal", LinearRegression())

Línea base      MAE:     9862   RMSE:    13612   R²: -0.008
Reg. lineal     MAE:     4177   RMSE:     5956   R²: 0.807


In [15]:
from sklearn.ensemble import RandomForestRegressor

bosque = evaluar("Random Forest", RandomForestRegressor(random_state=42))

Random Forest   MAE:     2583   RMSE:     4673   R²: 0.881


In [16]:
import joblib

for nombre, pipe in [("lineal", lineal), ("bosque", bosque)]:
    joblib.dump(pipe, f"prueba_{nombre}.joblib")
    print(nombre, round(os.path.getsize(f"prueba_{nombre}.joblib") / 1e6, 3), "MB")

lineal 0.004 MB
bosque 9.735 MB


In [17]:
bosque_chico = evaluar("RF profundidad 5",
                       RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42))

joblib.dump(bosque_chico, "prueba_bosque_chico.joblib")
print(round(os.path.getsize("prueba_bosque_chico.joblib") / 1e6, 3), "MB")

RF profundidad 5 MAE:     2419   RMSE:     4245   R²: 0.902
0.951 MB


In [18]:
from sklearn.model_selection import cross_val_score

for nombre, pipe in [("Reg. lineal", lineal), ("RF completo", bosque), ("RF profundidad 5", bosque_chico)]:
    mae = -cross_val_score(pipe, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(f"{nombre:18} MAE promedio: {mae.mean():6.0f}  ± {mae.std():.0f}")

Reg. lineal        MAE promedio:   4200  ± 89
RF completo        MAE promedio:   2671  ± 146
RF profundidad 5   MAE promedio:   2513  ± 162


Despues de realizar la validacion cruzada de los 3 modelos, la decision es que el modelo bosque es mejor, ademas se realizo un bosque limitado a 5 niveles, dando mejores resultados y evitando overfitting ya que de esa forma solo puede aprender patrones generales y no memoriza.

In [19]:
import sklearn

RUTA_MODELO = os.path.join("..", "app", "modelo", "modelo.joblib")
os.makedirs(os.path.dirname(RUTA_MODELO), exist_ok=True)

joblib.dump(bosque_chico, RUTA_MODELO)

print("Guardado en         :", RUTA_MODELO)
print("Versión scikit-learn:", sklearn.__version__)
print(f"Peso del archivo    : {os.path.getsize(RUTA_MODELO) / 1e6:.3f} MB")

Guardado en         : ..\app\modelo\modelo.joblib
Versión scikit-learn: 1.9.1
Peso del archivo    : 0.951 MB


In [22]:
cargado = joblib.load(RUTA_MODELO)

persona = pd.DataFrame([{"edad": 35, "sexo": "male", "imc": 28.5,
                         "hijos": 2, "fumador": "no", "region": "northwest"}])
print("Costo estimado:", round(cargado.predict(persona)[0], 2))

Costo estimado: 6792.2
